# rewrite

> Number the sentences of a Markdown file, a notebook or a Python file's docstrings for language models, compare the rewrites they suggest, and put rewrites into the file


In [ ]:
#| default_exp rewrite

In [ ]:
#| hide
from nbdev.showdoc import *

NB: this file is designed for an LLM reader, not a human. Add it to your LLM's context and ask it to explain.

- numbered: sentences, every newline ends one: `.md` whole text; `.py` docstrings; `.ipynb` Markdown cells, code-cell docstrings; headings, quotes, code, code cells, tables, front matter: unnumbered context
- `await rewrite_review(path)`: two models suggest rewrites, file untouched; you choose: `review_apply(path, {n: 'new text'})` or edit by hand
- `review_passed(path)`: sentences no model flagged: read them too    /models miss some
- `template=`: `prompt()` prose for people; `prompt('terse')` short reference lines, e.g. a page per primitive; `prompt('llm')` files an LLM reads
- needs `pip install slopometer[ai]`


In [ ]:
#| export
import ast, asyncio, itertools, mdhtml, textwrap
from bisect import bisect_right
from difflib import SequenceMatcher
from importlib.resources import files
from fastcore.utils import *
from fastcore.meta import delegates
from fastcore.tools import lnhash_at
from fastcore.nbio import read_nb, write_nb, validate_nb
from fastcore.xdg import xdg_state_home
from slopometer.segment import segment, _item
from slopometer.ai import sentences, SYSTEM, MODEL

In [ ]:
from fastcore.test import *
from nbdev.config import get_config
from fastcore.nbio import new_nb, mk_cell
import tempfile

## Markdown in reading order

A docstring is Markdown. `md_parts` lists what a reader of its prose sees, in order, as pairs. Each sentence of a paragraph or list item comes as a `Sent` paired with the list marker in front of it, such as `- ` or `2. `. The marker is an empty string when the sentence does not start a list item. A heading, block quote, code block or table comes as None paired with its text. The model needs these to follow the document, but it is not asked to rewrite them.

In [ ]:
#| export
_ctx = ('block_quote', 'code_block', 'table')

def _markup(txt):
    "Offsets of `txt`'s inline markup: each character of a link, image, emphasis, HTML tag or formula outside its text and code, spaces included and newlines left out"
    ns = mdhtml.inlines(txt)
    shown = {i for n in ns if n['type'] in ('text', 'code') for i in range(n['start'], n['end'])}
    return {i for n in ns if n['type'] not in ('text', 'code') for i in range(n['start'], n['end']) if i not in shown and txt[i] != '\n'}

def _whole(txt, mk, a, e, lo):
    "Offsets `a`,`e` of a sentence in `txt`, widened over the markup that touches it; `mk` is `_markup(txt)` and `lo` the lowest start allowed"
    while a > lo and a-1 in mk: a -= 1
    while e < len(txt) and e in mk: e += 1
    return a,e

def _visible(txt):
    "The copy of `txt` that `sentences` splits, with every offset and line where it was: each formula's characters other than whitespace become X, so that a formula reads as words in its sentence"
    out = list(txt)
    for n in mdhtml.inlines(txt):
        if n['type'] == 'math_inline': out[n['start']:n['end']] = [c if c.isspace() else 'X' for c in txt[n['start']:n['end']]]
    return ''.join(out)

def md_parts(txt):
    "Markdown `txt` in reading order as pairs: a `Sent` with its list marker, or None with the text of a heading, block quote, code block, table or YAML front matter"
    vis = _visible(txt)
    blocks = segment(vis)
    prose = [b for b in blocks if b.kind != 'heading']
    lines = txt.split('\n')
    offs = list(itertools.accumulate((len(l)+1 for l in lines), initial=0))
    res,end,para,lo,mk = [(b.start, None, txt[b.start:b.start+len(b.txt)]) for b in blocks if b.kind == 'heading'],0,None,0,_markup(txt)
    if fm := re.match(r'---\n.*?\n---[ \t]*(?=\n|\Z)', txt, re.S): res.append((0, None, fm[0]))
    for b in mdhtml.blocks(vis, nested=True):
        if b['type'] not in _ctx or b['start'] < end: continue
        end = b['end']
        res.append((offs[b['start']], None, '\n'.join(lines[b['start']:end]).rstrip()))
    for s in sentences(vis, 1):
        b = prose[s.para]
        m = _item.match(b.txt) if b.kind == 'item' and s.para != para else None
        if s.para != para: lo = b.start + (m.end() if m else 0)
        if s.para != para: col = lo - txt.rfind('\n', 0, b.start) - 1
        s.start,s.end = _whole(txt, mk, s.start, s.end, lo)
        ls = txt.rfind('\n', 0, s.start) + 1
        s.indent = ' '*max(0, s.start-ls-col) if ls > b.start and not txt[ls:s.start].strip() else ''
        s.text,para,lo = txt[s.start:s.end],s.para,s.end
        res.append((s.start, s, m[0] if m else ''))
    return [o[1:] for o in sorted(res, key=lambda o: o[0])]

This docstring text has a heading, a block quote, a numbered list with a two-line item, a link, and a fenced block:

In [ ]:
MD = """Widgets for the shop floor.

# Shipping

> Ship it today.

1. Pack the widget.
   Tape the box shut.
2. Call [`ship`](api.md#ship)

```python
ship(w)
```
"""
parts = md_parts(MD)
parts

Each `Sent` has the offsets of its sentence in the text. A newline always ends a sentence. `md_parts` keeps the spacing inside a line, which preserves the columns in a file of aligned rows. `md_parts` keeps each link, emphasis, HTML tag and formula whole inside its sentence, even a link target with spaces in it. The model sees them and can keep them in its rewrite:

In [ ]:
FENCE = '`'*3
test_eq([(s and s.text, t) for s,t in parts], [('Widgets for the shop floor.', ''), (None, '# Shipping'), (None, '> Ship it today.'),
    ('Pack the widget.', '1. '), ('Tape the box shut.', ''), ('Call [`ship`](api.md#ship)', '2. '), (None, FENCE+'python\nship(w)\n'+FENCE)])
tape = parts[4][0]
test_eq(MD[tape.start:tape.end], 'Tape the box shut.')
test_eq([s.text for s,_ in md_parts('fork    exact clone\nagent   starts with nothing\n')], ['fork    exact clone', 'agent   starts with nothing'])
sp = 'Open [My Report.md](</a/My Project/My Report.md:3>)\n<span class="x">Then</span> close it.'
test_eq([s.text for s,_ in md_parts(sp)], ['Open [My Report.md](</a/My Project/My Report.md:3>)', '<span class="x">Then</span> close it.'])
test_eq([s.text for s,_ in md_parts('\\(x + y\\) is positive.\nThe total is \\(a + b\\)\n')], ['\\(x + y\\) is positive.', 'The total is \\(a + b\\)'])
test_eq(md_parts('# Sum \\(a + b\\)\nIt grows.')[0], (None, '# Sum \\(a + b\\)'))

## Docstrings in a Python file

This source has a module docstring with a heading, a block quote and a numbered list. It also has a function with a one-line docstring, a method with an indented docstring of two paragraphs, and a method with none:

In [ ]:
SRC = r'''"""Widgets for the shop floor.

# Shipping

`make` builds a widget. Finished widgets ride to the customer with `ship`.

> Ship it today.

1. Pack the widget.
2. Call `ship`.
"""
def make(spec):
    "Build a widget from `spec`"
    return Widget(spec)

class Widget:
    def __init__(self, spec): self.spec = spec
    def ship(self, addr):
        r"""Post the widget to `addr`.

        The label carries the order number. It raises `ValueError` when `addr` is empty.
        """
        if not addr: raise ValueError(addr)
'''

The docstrings to find belong to the module, its functions and its classes. `_named` walks the syntax tree and gives each function and class a dotted name, such as `Widget.ship` for a method. `_doclit` returns the string literal that is a node's docstring, or None when the node has no docstring.

In [ ]:
#| export
_defs = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)

def _doclit(n):
    "The string literal that is the docstring of node `n`, or None"
    b = n.body
    if b and isinstance(b[0], ast.Expr) and isinstance(c := b[0].value, ast.Constant) and isinstance(c.value, str): return c

def _named(node, pre=''):
    "The function and class nodes under `node` in source order, each with its dotted name"
    for n in ast.iter_child_nodes(node):
        if isinstance(n, _defs):
            yield pre+n.name, n
            yield from _named(n, f'{pre}{n.name}.')
        else: yield from _named(n, pre)

def _docnodes(tree): return [(None, tree), *_named(tree)]

In [ ]:
tree = ast.parse(SRC)
test_eq([nm for nm,_ in _named(tree)], ['make', 'Widget', 'Widget.__init__', 'Widget.ship'])
[(nm, bool(_doclit(n))) for nm,n in _docnodes(tree)]

`py_docstrings` returns each docstring of a Python source as a name, the text, and the offset of that text in the source. The module docstring comes first, with the name None. Each function and class follows in source order. A method has a dotted name such as `Widget.ship`. The text is taken from the file as written, between the quotes, because a rewrite is later put back at the same offsets.

In [ ]:
#| export
def py_docstrings(src):
    "Each docstring in Python source `src` as (name, text, offset): the module's first, with name None, then each function's and class's"
    lines = src.split('\n')
    starts = list(itertools.accumulate((len(l)+1 for l in lines), initial=0))
    def off(ln, col): return starts[ln-1] + len(lines[ln-1].encode()[:col].decode())  # ast columns count UTF-8 bytes
    res = []
    for nm,n in _docnodes(ast.parse(src)):
        if not (c := _doclit(n)): continue
        a = off(c.lineno, c.col_offset)
        m = re.match(r'''[a-zA-Z]*("""|\'\'\'|"|')''', src[a:])
        res.append((nm, src[a+m.end():off(c.end_lineno, c.end_col_offset)-len(m[1])], a+m.end()))
    return res

In [ ]:
ds = py_docstrings(SRC)
for nm,t,o in ds: test_eq(SRC[o:o+len(t)], t)
[(nm, t) for nm,t,o in ds]

The later lines of a docstring are usually indented to match the code around them. Markdown would read a paragraph indented by four spaces as code. `_dedent` removes the indentation that those lines share. It also returns a function that turns an offset in the result into the offset of the same character in the original:

In [ ]:
#| export
def _dedent(t):
    "`t` without the indentation that its later lines share, and a function that maps an offset in the result to its offset in `t`"
    lines = t.splitlines(keepends=True)
    ws = [len(l)-len(l.lstrip(' \t')) for l in lines]
    ind = min((w for l,w in zip(lines[1:], ws[1:]) if l.strip()), default=0)
    cuts = [0] + [min(ind, w) for w in ws[1:]]
    dst,ost = (list(itertools.accumulate(ns, initial=0)) for ns in ([len(l)-c for l,c in zip(lines, cuts)], map(len, lines)))
    def back(p):
        i = min(bisect_right(dst, p), len(lines)) - 1
        return ost[i] + cuts[i] + p - dst[i]
    return ''.join(l[c:] for l,c in zip(lines, cuts)), back

In [ ]:
d,back = _dedent('First.\n\n    Second line.\n    ')
test_eq(d, 'First.\n\nSecond line.\n')
test_eq(back(d.index('Second')), 12)
d

`py_docs` splits each docstring into the parts that `md_parts` returns. The offsets of each sentence are positions in the whole source.

In [ ]:
#| export
def py_docs(src):
    "Each docstring of Python source `src` as (name, parts): parts are as `md_parts` returns, with sentence offsets that are positions in `src`"
    res = []
    for nm,t,o in py_docstrings(src):
        d,back = _dedent(t)
        parts = md_parts(d)
        for s,_ in parts:
            if s: s.start,s.end = o+back(s.start), o+back(s.end-1)+1
        if parts: res.append((nm, parts))
    return res

In [ ]:
docs = py_docs(SRC)
docs

The second paragraph of the method's docstring is found, though the file indents it. The offsets of each `Sent` are positions in the source file:

In [ ]:
test_eq([nm for nm,_ in docs], [None, 'make', 'Widget.ship'])
raises = docs[2][1][2][0]
test_eq(raises.text, 'It raises `ValueError` when `addr` is empty.')
test_eq(SRC[raises.start:raises.end], 'It raises `ValueError` when `addr` is empty.')

## The input for the model

`rewrite_input` writes the text that the model reads. Each docstring starts with a line that names it: `Module docstring:`, or the name of the function or class. Prose that is not a docstring has the name `''`. It gets no such line in a Markdown file. A notebook's code cells have the name `CODE`. Each is shown as fenced code with no number, and a `Markdown cell:` line marks the prose that follows it. Every sentence has a number in square brackets. The numbers do not restart for each docstring, because an answer names a sentence by its number alone. A list marker stays in front of the number of the sentence that starts the item. A line indented further than the first line of its paragraph is shown with that extra indentation after its number. Aligned rows then keep their columns. Headings, block quotes, code blocks and tables have no number.

A module can keep part of its docstring elsewhere and leave a `{name}` placeholder for it, as `aai_coding.write_docs` does. Pass that text in `fills`. The line after each sentence with a placeholder then gives the text that the placeholder stands for.


In [ ]:
#| export
CODE = '#code'

def rewrite_input(
    docs, # (name, parts) pairs, as `file_docs` returns
    fills=None, # Text that each `{name}` placeholder in the sentences stands for
    skip=None, # Regex; a paragraph or list item that it matches gets `[-]` in place of each number
    nums=True, # Number the sentences? False gives the same text with no numbers, as `_related` shows an earlier version
    mod_only=False, # Give `[-]` to the docstrings of functions and classes? For a pyskill, whose module docstring is the skill's text
):
    "The text a model reads for `docs`: every sentence numbered unless `skip` matches its paragraph or `mod_only` excludes its docstring, with each docstring under a line that names it, and each code cell shown as context"
    res,i,fills,last = [],0,fills or {},''
    for nm,parts in docs:
        if nm not in ('', CODE): res += ['Module docstring:' if nm is None else f'Docstring of `{nm}`:', '']
        elif nm == '' and last != '': res += ['Markdown cell:', '']
        last = nm
        prev,paras = None,{}
        for s,_ in parts:
            if s: paras[s.para] = f"{paras.get(s.para, '')} {s.text}".strip()
        skips = set(paras) if mod_only and nm not in (None, '', CODE) else {p for p,t in paras.items() if skip and re.search(skip, t)}
        for k,(s,t) in enumerate(parts):
            if k and not (s and prev and (s.para == prev.para or t)): res.append('')
            res.append(t if not s else f"{t}{s.indent}{s.text}" if not nums else f"{t}[{'-' if s.para in skips else i}] {s.indent}{s.text}")
            if s:
                res += [f'  ({{{ph}}} stands for: {fills[ph]})' for ph in re.findall(r'\{(\w+)\}', s.text) if ph in fills]
                i += 1
            prev = s
        res.append('')
    return '\n'.join(res).rstrip()

In [ ]:
inp = rewrite_input(docs)
print(inp)

In [ ]:
for o in ('Module docstring:', '# Shipping', '> Ship it today.', '2. [4] Call `ship`.', 'Docstring of `Widget.ship`:', '[8] It raises `ValueError` when `addr` is empty.'):
    assert o in inp.splitlines()
test_eq(rewrite_input([('', parts)]).splitlines()[0], '[0] Widgets for the shop floor.')
filled = rewrite_input([('', md_parts('Avoid these.\n\n1. {hedging}'))], dict(hedging='Hedging. Say what is true.'))
test_eq(filled.splitlines()[-2:], ['1. [1] {hedging}', '  ({hedging} stands for: Hedging. Say what is true.)'])
test_eq(rewrite_input([('', md_parts('fork    exact clone\n        tell: its part\n'))]).splitlines(), ['[0] fork    exact clone', '[1]         tell: its part'])

Some lines are reference text that no model should rewrite, such as the line of key bindings at the top of each page of a manual. Pass `skip`, a regex, to leave them unnumbered. A paragraph or list item that `skip` matches gets `[-]` in place of each number, so the model reads it and has no number to answer with. The other sentences keep their numbers:

In [ ]:
skipped = rewrite_input([('', parts)], skip=r'^Pack').splitlines()
for o in ('1. [-] Pack the widget.', '[-] Tape the box shut.', '2. [3] Call [`ship`](api.md#ship)'): assert o in skipped
skipped

A pyskill's module docstring is the text of the skill, and its other docstrings document code. `mod_only=True` gives the sentences of those other docstrings `[-]`, so the model reads them and does not rewrite them. `nums=False` leaves out every number. `_related` uses it to show an earlier version of a file in the same form as the document:

In [ ]:
mo = rewrite_input(docs, mod_only=True).splitlines()
for o in ('[0] Widgets for the shop floor.', '[-] Build a widget from `spec`', '[-] The label carries the order number.'): assert o in mo
plain = rewrite_input(docs, nums=False)
assert '\n1. Pack the widget.' in plain and '[' not in plain
print(plain)

## The model's answer

The prompt asks for one answer per sentence. `7 -` keeps sentence 7. `7 <text>` replaces it. `7-8 <text>` replaces sentences 7 and 8 together. `7 ? <question>` asks what the sentence means. `7 ? cut: <why>` and `7 ? lost: <fact>` are questions too, so they need no parsing of their own. `parse_rewrites` reads those answers into (first, last, text) triples. It accepts the number with or without brackets. A line that starts with neither a number nor a space belongs to no answer, and it is skipped.

In [ ]:
#| export
_ans = re.compile(r'(?![ \t])[^\w\n]*(\d+)(?:[ \t]*-[ \t]*(\d+))?[^\w\n]*?(?:[ \t]+(?:>[ \t]+)?(\S.*?))?[ \t]*')
def parse_rewrites(reply):
    "The answers of `reply` as (first, last, text) triples: text is `-` to keep the sentences, starts with `?` for a question, and is otherwise the rewrite, without the `> ` that a model may repeat from a description; indented lines under an answer are rows of its text"
    res,rows = [],None
    for l in reply.splitlines():
        if m := _ans.fullmatch(l):
            rows = []
            res.append((int(m[1]), int(m[2] or m[1]), m[3], rows))
        elif rows is not None and l[:1] in (' ', '\t'): rows.append(l.rstrip())
        elif l.strip(): rows = None
    return [(a, b, '\n'.join([t]+rs) if t else textwrap.dedent('\n'.join(rs))) for a,b,t,rs in res if t or rs]

In [ ]:
reply = """0 -
1 -
2 Call `ship` to post a finished widget to the customer.
[3] -
4 -
5 -
6 -
7 The order number is printed on the label.
8 ? what does it do when `addr` is only spaces"""
ans = parse_rewrites(reply)
test_eq(ans[2:4], [(2, 2, 'Call `ship` to post a finished widget to the customer.'), (3, 3, '-')])
test_eq(parse_rewrites('Here are my answers.\n5-6 One sentence for two.'), [(5, 6, 'One sentence for two.')])
ans[-2:]

A model can answer a run with rows, when a table or a list says it better than a sentence. It writes the number, then each row on a line of its own, indented. `parse_rewrites` keeps those lines in the answer. With no text beside the number, the rows are the whole answer:

In [ ]:
rows = parse_rewrites("""3 -
4-6 Read through the kernel:
    file    `view_file`
    search  `rg`
7-8
    always        `ws-sync`
    code changed  restart
9 -""")
test_eq(rows[1], (4, 6, 'Read through the kernel:\n    file    `view_file`\n    search  `rg`'))
test_eq(rows[2], (7, 8, 'always        `ws-sync`\ncode changed  restart'))
rows[1:3]

## Putting the rewrites into the source

A rewrite must not damage the file. `same_code` makes a check for that. The check compares the syntax tree of a new source with the original's and ignores what the docstrings say. A quotation mark that ends a docstring early is one way to fail it:

In [ ]:
#| export
def _code(src):
    "The syntax tree of Python source `src` as text, with every docstring emptied; None when `src` does not parse"
    try: tree = ast.parse(src)
    except SyntaxError: return None
    for _,n in _docnodes(tree):
        if c := _doclit(n): c.value = ''
    return ast.dump(tree)

def same_code(src):
    "A function that tells whether another source has the same code as Python source `src`, whatever its docstrings say"
    code = _code(src)
    return lambda s: _code(s) == code

In [ ]:
ok = same_code(SRC)
assert ok(SRC.replace('Build a widget', 'Make one widget'))
assert not ok(SRC.replace('return Widget(spec)', 'return None'))
assert not ok(SRC.replace('Build a widget', 'Build a "widget"'))

`Rewrites` applies the answers. Its `new` attribute is the source with each rewrite in place of the old sentence. `rows` lists every rewrite and question as (first, last, kind, old text, new text), for a person to review. `missing` counts the sentences that got no answer line. An answer that repeats its sentence unchanged counts as `-`.

In [ ]:
#| export
_lead = re.compile(r'[ \t]*(?:[-*+>]|\d+[.)])?[ \t]*')
def _cut(src, ss, a, b, back):
    "Start and end of the text of `src` that deleting sentences `a` to `b` of `ss`, (docs index, `Sent`) pairs, removes. The space up to the next sentence of their paragraph goes too, or else the space back to the previous one, and a neighbour on the same line comes first. A whole paragraph takes its lines and the line breaks on one side, so its neighbours keep the wider gap, a text keeps its final line break, and no blank line is left at the start of a text or cell; no part of `SEP` goes. `back` holds the first sentence of each run already deleted with the space before it."
    (k,s0),e = ss[a],ss[b][1].end
    st = s0.start
    def near(i): return 0 <= i < len(ss) and ss[i][0] == k and ss[i][1].para == s0.para
    nxt = ss[b+1][1].start if near(b+1) and b+1 not in back else None
    prv = ss[a-1][1].end if near(a-1) else None
    for ws in (' \t', ' \t\n'):
        if nxt is not None and not src[e:nxt].strip(ws): return st,nxt
        if prv is not None and not src[prv:st].strip(ws): return prv,e
    x,y = src.rfind('\n', 0, st)+1, src.find('\n', e)
    if y < 0: y = len(src)
    if src[e:y].strip() or not _lead.fullmatch(src[x:st]): return st,e
    nb,na = 0,0
    while x-nb and src[x-nb-1] == '\n' and not src.endswith(SEP, 0, x-nb): nb += 1
    while y+na < len(src) and src[y+na] == '\n' and not src.startswith(SEP, y+na): na += 1
    first,last = x-nb == 0 or src.endswith(SEP, 0, x-nb), y+na == len(src) or src.startswith(SEP, y+na)
    return (x-nb, y+na) if first else (x-nb, y) if last or na > nb else (x, y+na)
class Rewrites:
    def __init__(self,
        src, # The text that the sentence offsets are positions in
        docs, # (name, parts) pairs, as `py_docs` returns
        answers, # (first, last, text) triples, as `parse_rewrites` returns
        valid=None, # Called with the source after each rewrite; a false result leaves that rewrite out
    ):
        ss = [(k, s) for k,(_,ps) in enumerate(docs) for s,_ in ps if s]
        self.src,self.n,self.rows,done,lo,back = src,len(ss),[],set(),len(ss),set()
        for a,b,t in sorted(answers, reverse=True):
            if not 0 <= a <= b < len(ss): continue
            done.update(range(a, b+1))
            old = ' '.join(s.text for _,s in ss[a:b+1])
            if t in ('-', old): continue
            if t.startswith('?'): kind,t = 'QUESTION',t[1:].strip()
            else:
                t = t if t.strip() else ''
                x,y = (ss[a][1].start, ss[b][1].end) if t else _cut(src, ss, a, b, back)
                new = src[:x] + t + src[y:]
                if b >= lo or len({(k, s.para) for k,s in ss[a:b+1]}) > 1 or (valid and not valid(new)): kind = 'LEFT OUT'
                else:
                    kind,src,lo = 'AFTER',new,a
                    if x < ss[a][1].start: back.add(a)
            self.rows.append((a, b, kind, old, t))
        self.new,self.missing = src,len(ss)-len(done)
        self.rows.reverse()

With the answers above, two sentences are rewritten. The rest of the file is as it was:

In [ ]:
r = Rewrites(SRC, docs, ans, valid=same_code(SRC))
test_eq([(a, kind) for a,_,kind,*_ in r.rows], [(2, 'AFTER'), (7, 'AFTER'), (8, 'QUESTION')])
assert '`make` builds a widget. Call `ship` to post a finished widget to the customer.\n' in r.new
assert '        The order number is printed on the label. It raises `ValueError` when `addr` is empty.\n' in r.new
assert same_code(SRC)(r.new)
print(r.new)

`of` returns the rows of one kind. The display gives the counts, then each row as the old text and the new:

In [ ]:
#| export
@patch
def of(self:Rewrites, kind):
    "The rows of one kind: 'AFTER' for a rewrite made, 'QUESTION', or 'LEFT OUT' for a rewrite not made"
    return [r for r in self.rows if r[2] == kind]

@patch
def __repr__(self:Rewrites):
    hdr = f"{self.n} sentences. Rewrites: {len(self.of('AFTER'))}. Questions: {len(self.of('QUESTION'))}."
    if out := self.of('LEFT OUT'): hdr += f' Left out: {len(out)}.'
    if self.missing: hdr += f' No answer: {self.missing}.'
    def row(a, b, kind, old, t):
        n = a if a == b else f'{a}-{b}'
        return f'{n} BEFORE: {old}\n{n} {kind}: {t}'
    return '\n\n'.join([hdr] + [row(*r) for r in self.rows])

In [ ]:
test_eq(len(r.of('AFTER')), 2)
r

`Rewrites` leaves a rewrite out in three cases. Its range runs across two docstrings or two paragraphs. Another rewrite already covers one of its sentences. Or the `valid` check fails. Here the first answer joins a list item of the module docstring to the docstring of `make`, the second puts a double quotation mark into a docstring that double quotation marks enclose, and the last two both cover sentence 2:

In [ ]:
out = Rewrites(SRC, docs, [(4, 5, 'Call `ship` and build a widget.'), (5, 5, 'Build a "widget" from `spec`'), (2, 2, 'Use `ship`.'), (1, 2, 'Use `make` and `ship`.')],
    valid=same_code(SRC))
test_eq([(a, b, kind) for a,b,kind,*_ in out.rows], [(1, 2, 'LEFT OUT'), (2, 2, 'AFTER'), (4, 5, 'LEFT OUT'), (5, 5, 'LEFT OUT')])
test_eq(out.missing, 5)
out

## Notebooks

A notebook holds prose in its Markdown cells and docstrings in its code cells. `nb_docs` reads both in cell order. A Markdown cell gives its prose under the name `''`. A code cell gives two things. Its whole source comes first under the name `CODE`, as context that the model reads and does not answer. The docstring of each function and class in it follows, numbered. A code cell that Python cannot parse, such as one with a magic, gives its source only. Raw cells and empty cells give nothing.

In an nbdev notebook the block quote under the title is the module's description. `nb_docs` numbers its sentences so that the model reviews them, though other block quotes are context only. A description keeps its `> ` marker in the model's input. The prompt tells the model to keep such a sentence apart from the text after it, because listings show the description on its own.

A `Rewrites` works on one text. `nb_docs` therefore also returns the cell sources joined by `SEP`, and the sentence offsets are positions in that text. `_addr` gives each sentence an `addr`, the exhash address of the line it starts on:

In [ ]:
#| export
SEP = '\n\x00\n'

def _addr(src, docs, pre=''):
    "Set `addr` on each sentence of `docs` to `pre` and the exhash address of the line of `src` that it starts on; returns `docs`"
    for _,ps in docs:
        for s,_ in ps:
            if s: s.addr = pre + lnhash_at(src, src[:s.start].count('\n')+1)
    return docs

A block quote directly under a title is the description of the page, so `_md_parts` numbers its sentences, whereas any other block quote is context. `_cell_docs` reads one cell, and `nb_docs` puts the cells together. In a notebook, a sentence's `addr` starts with its cell ID:

In [ ]:
#| export
def _desc(src):
    "Markdown `src` with the `> ` of the description under its title, or under the `NB:` line that follows the title, turned into spaces, which keeps every offset and makes the description a paragraph"
    m = re.match(r'(?s:---\n.*?\n---[ \t]*\n+)?# .*\n+(?:NB: .*\n+)?((?:> .*\n?)+)', src)
    return src[:m.start(1)] + re.sub(r'(?m)^> ', '  ', m[1]) + src[m.end(1):] if m else src

def _md_parts(src):
    "`md_parts` of Markdown `src`, with the description under its title as sentences that have the marker `> `"
    return [(s, '> ' if s and src.startswith('> ', src.rfind('\n', 0, s.start)+1) else t) for s,t in md_parts(_desc(src))]

def _cell_docs(c):
    "The (name, parts) pairs of notebook cell `c`, with offsets that are positions in its source; a code cell's source comes first, as context under the name `CODE`"
    src = c.source
    if c.cell_type == 'markdown': return [('', _md_parts(src))]
    if c.cell_type != 'code' or not src.strip(): return []
    res = [(CODE, [(None, f'```python\n{src}\n```')])]
    try: return res + [o for o in py_docs(src) if o[0] is not None]
    except SyntaxError: return res

def nb_docs(nb):
    "The cell sources of notebook `nb` joined by `SEP`, and its (name, parts) pairs: the prose of each Markdown cell, and each code cell's source followed by its docstrings; each sentence has its `addr` as cell ID and line address"
    res,off = [],0
    for c in nb.cells:
        for nm,parts in _addr(c.source, _cell_docs(c), f'{c.id}:'):
            for s,_ in parts:
                if s: s.start,s.end = s.start+off,s.end+off
            if parts: res.append((nm, parts))
        off += len(c.source)+len(SEP)
    return SEP.join(c.source for c in nb.cells), res

This notebook has a title cell with a description, an exported function, a prose cell, a cell with a magic, and a raw cell:

In [ ]:
NB = new_nb([mk_cell('# widgets\n\n> Build widgets and ship them to customers.', 'markdown'),
    mk_cell('#| export\ndef make(spec):\n    "Build a widget from `spec`"\n    return Widget(spec)'),
    mk_cell('`make` needs a spec. Finished widgets ride to the customer.', 'markdown'),
    mk_cell('%time make(1)'), mk_cell('Not prose.', 'raw')])
nsrc,ndocs = nb_docs(NB)
ndocs

In the input for the model, the description has its `> ` marker, each code cell is fenced, and a `Markdown cell:` line marks where prose starts again. A model may repeat the `> ` in its answer, and `parse_rewrites` drops it:

In [ ]:
test_eq([nm for nm,_ in ndocs], ['', CODE, 'make', '', CODE])
desc = ndocs[0][1][1][0]
test_eq(nsrc[desc.start:desc.end], 'Build widgets and ship them to customers.')
test_eq(desc.addr, f'{NB.cells[0].id}:{lnhash_at(NB.cells[0].source, 3)}')
ninp = rewrite_input(ndocs)
assert '> [0] Build widgets and ship them to customers.' in ninp
assert '    return Widget(spec)\n```\n\nDocstring of `make`:\n\n[1] Build a widget from `spec`\n\nMarkdown cell:\n\n[2] `make` needs a spec.' in ninp
assert ninp.endswith('```python\n%time make(1)\n```')
test_eq(parse_rewrites('0 > Build and ship widgets.'), [(0, 0, 'Build and ship widgets.')])
print(ninp)

A rewrite in a notebook must leave the code of every code cell as it was. `_same_cells` makes that check for a `SEP`-joined text:

In [ ]:
#| export
def _same_cells(nb):
    "A function that tells whether a `SEP`-joined text has the same code as `nb` in each code cell, whatever the docstrings say"
    idx = [i for i,c in enumerate(nb.cells) if c.cell_type == 'code']
    def codes(ss): return [_code(ss[i]) for i in idx]
    orig = codes([c.source for c in nb.cells])
    return lambda s: codes(s.split(SEP)) == orig

In [ ]:
nok = _same_cells(NB)
nr = Rewrites(nsrc, ndocs, [(0, 0, 'Build widgets and ship them.'), (1, 1, 'Build a "widget"'), (3, 3, 'We send finished widgets to the customer.')], valid=nok)
test_eq([(a, kind) for a,_,kind,*_ in nr.rows], [(0, 'AFTER'), (1, 'LEFT OUT'), (3, 'AFTER')])
test_eq(nr.new.split(SEP)[0], '# widgets\n\n> Build widgets and ship them.')
nr

## Rewriting a file

`file_docs` reads a file and returns what the later steps need: the text, the (name, parts) pairs, and the check that `Rewrites` runs after each rewrite. A notebook gives the text and pairs that `nb_docs` returns, with the `_same_cells` check. A `.py` file gives its docstrings and the `same_code` check. Any other file is read as Markdown and has no check. Its description is numbered too, as in the summary line under the title of an `llms.txt` file. Each sentence also gets an `addr`: the exhash address of the line it starts on, which in a notebook follows the cell ID.

`write_new` writes the `new` text of a `Rewrites` to a file. For a notebook it reads the notebook at `path` and replaces the source of each cell.


In [ ]:
#| export
def file_docs(path):
    "The text of the file at `path`, its (name, parts) pairs with a line address as each sentence's `addr`, and a `valid` check for `Rewrites`: a notebook's cells, a `.py` file's docstrings, or the whole text of any other file"
    path = Path(path).expanduser()
    if path.suffix == '.ipynb':
        nb = read_nb(path)
        return *nb_docs(nb), _same_cells(nb)
    src = path.read_text()
    if path.suffix == '.py': return src, _addr(src, py_docs(src)), same_code(src)
    return src, _addr(src, [('', _md_parts(src))]), None

def write_new(path, new, dest=None):
    "Write `new`, the `new` text of a `Rewrites` for the file at `path`, to `dest`, or over `path` when `dest` is None; a notebook gets it as its cell sources and is validated first"
    path = Path(path).expanduser()
    dest = Path(dest).expanduser() if dest else path
    if path.suffix != '.ipynb': return dest.write_text(new)
    nb = read_nb(path)
    for c,s in zip(nb.cells, new.split(SEP), strict=True): c.source = s
    write_nb(validate_nb(nb), dest)

In [ ]:
root = get_config().config_path
src,fdocs,valid = file_docs(root/'README.md')
test_eq([nm for nm,_ in fdocs], [''])
test_is(valid, None)
src,fdocs,valid = file_docs(root/'slopometer'/'score.py')
assert valid(src) and fdocs[0][0] is None
first = fdocs[0][1][0][0]
test_eq(first.addr, lnhash_at(src, src[:first.start].count('\n')+1))
first.addr

A Markdown file's description is numbered, and a block quote further down is not. The description can follow the `NB:` line that a file for a language model has under its title. YAML front matter is shown without a number:

In [ ]:
f = Path(tempfile.mkdtemp())/'llms.txt'
f.write_text('---\nname: widgets\n---\n# Widgets\n\nNB: this file is designed for an LLM reader, not a human.\n\n> Widget API. Most shop idioms fail here.\n\n- make: `make(spec)`.\n\n> A later quotation.\n')
inp = rewrite_input(file_docs(f)[1]).splitlines()
for o in ('name: widgets', '> [1] Widget API.', '> [2] Most shop idioms fail here.', '> A later quotation.'): assert o in inp
inp

A notebook gives the check for its code cells, and each `addr` starts with a cell ID:

In [ ]:
src,fdocs,valid = file_docs(root/'nbs'/'00_core.ipynb')
assert valid(src) and fdocs[0][1][1][0].addr.startswith(read_nb(root/'nbs'/'00_core.ipynb').cells[0].id+':')
rewrite_input(fdocs).splitlines()[:3]

`prompt` returns the prompt that ships with the package. It has a `{document}` placeholder where the numbered sentences go. The default prompt teaches full explanatory sentences. `prompt('terse')` is for reference pages written as short lines, such as the page for each primitive of an array language. It keeps the default prompt's answer format and its last section, and has guidance and examples of its own between them. `prompt('llm')` is for files that a language model reads, such as an `llms.txt` file, a skill or a system prompt, written in the dense style that `aai_coding.write_llms` describes. Every style has two answers that are questions: `? cut:` for a sentence that the reader does not need, and `? lost:` for a fact that an earlier version has and this one lacks. `_related` writes out the files that a caller passes as `context`, for the model to consult. It puts the file passed as `earlier` last, under a heading that says it is an earlier version of the document. It shows that file as `rewrite_input` shows the document, without numbers. The model then compares like with like, and cannot report the code of a `.py` file as a fact that the docstrings have lost.


In [ ]:
#| export
def prompt(
    style=None, # Name of a style whose examples replace the default ones, `'terse'` or `'llm'`
):
    "The rewrite prompt that ships with the package, with a `{document}` placeholder for the numbered sentences"
    f = files('slopometer')
    res = f.joinpath('rewrite_prompt.txt').read_text()
    if not style: return res
    doc = '# The document'
    return res[:res.index('\n# ')+1] + f.joinpath(f'{style}_prompt.txt').read_text() + doc + res.split(doc)[1]

def _related(paths, earlier=None, fills=None):
    "The contents of each of `paths`, each under a heading that names the file and says that it needs no answer; then `earlier` as an earlier version of the document, in the form that `rewrite_input` gives the document, without numbers"
    def txt(f): return '\n\n'.join(c.source for c in read_nb(f).cells) if f.suffix == '.ipynb' else f.read_text()
    res = ''.join(f'\n\n# Related file `{f.name}`\n\nThis file is for reference and needs no answer.\n\n{txt(f)}' for f in (Path(o).expanduser() for o in listify(paths)))
    if not earlier: return res
    return res + f'\n\n# Earlier version of the document\n\nIt needs no answer. Use it only for `? lost:` answers.\n\n{rewrite_input(file_docs(earlier)[1], fills, nums=False)}'

Pass a style's prompt as `template`, as in `rewrite_review(path, template=prompt('terse'))`. Every style shares the default prompt's first and last sections, and the first has both question answers. The last lines check what `_related` writes for an earlier version. For a `.py` file that is its docstrings without numbers, and none of its code:

In [ ]:
p,hdr = prompt(),'# The document'
for style in 'terse','llm':
    t = prompt(style)
    test_eq(t.split('\n# ')[0], p.split('\n# ')[0])
    test_eq(t.split(hdr)[1], p.split(hdr)[1])
assert 'Never turn a terse line into a longer sentence.' in prompt('terse')
for o in '? cut:','? lost:': assert o in p.split('\n# ')[0]
assert _related(None, f).startswith('\n\n# Earlier version of the document') and 'Widget API' in _related(None, f)
(py := f.with_suffix('.py')).write_text(SRC)
early = _related(None, py)
assert '`make` builds a widget.' in early and 'return Widget' not in early and '[1]' not in early

`ask_reply` sends the whole file's input to a model in one call and returns the reply as text. It uses `prompt()` unless the caller passes another template. `rewrite_file` runs every step on one file and returns a `Rewrites`. It does not write the file.

In [ ]:
#| export
async def ask_reply(
    docs, # (name, parts) pairs, as `file_docs` returns
    template=None, # Prompt with a `{document}` placeholder; `prompt()` if None
    model=MODEL, # Model name for `fastllm`
    effort='low', # Reasoning effort
    system=SYSTEM, # System message; None sends none
    fills=None, # Text that each `{name}` placeholder in the sentences stands for
    context=None, # Files that the model can consult, such as the code that a document describes
    skip=None, # Regex; a paragraph or list item that it matches gets `[-]` in place of each number
    earlier=None, # File that holds an earlier version of the document, for `? lost:` answers
    mod_only=False, # Give `[-]` to the docstrings of functions and classes? For a pyskill, whose module docstring is the skill's text
):
    "`model`'s reply to `template` with the numbered sentences of `docs` in it, as text"
    from fastllm.acomplete import acomplete
    from fastllm.chat import contents
    from aidialog.msg_parts import mk_msgs
    msg = (template or prompt()).replace('{document}', rewrite_input(docs, fills, skip, mod_only=mod_only) + _related(context, earlier, fills))
    return contents(await acomplete(mk_msgs([msg]), model, system=system, reasoning_effort=effort)).text

@delegates(ask_reply)
async def rewrite_file(path, **kwargs):
    "`Rewrites` for the file at `path`, leaving out any rewrite that would change the code of a `.py` file or a notebook"
    src,docs,valid = file_docs(path)
    return Rewrites(src, docs, parse_rewrites(await ask_reply(docs, **kwargs)), valid=valid)

With `GEMINI_API_KEY` set, rewrite one file and write the result somewhere else:

In [ ]:
#| eval: false
rw = await rewrite_file(root/'slopometer'/'score.py')
write_new(root/'slopometer'/'score.py', rw.new, '/tmp/score.rewritten.py')
rw


## Two models, read by a person

A model's rewrite is a suggestion. It can drop a fact, rename a term or split a sentence that was better whole. In this process the models only point at sentences, and a person or an agent reads what they said and writes the final text. `compare` prepares that reading. It takes each model's answers under the model's name. The first line counts the sentences, and for each model the runs it rewrote and the questions it asked. Answers whose sentence ranges overlap are grouped into one run. For each run, it gives the address of the first sentence, the original text, and then each model's answer. Models whose answers differ only in case, spacing or punctuation share a line. A model that kept the sentences shows `-`.

With `now`, the sentences of the file after editing, each run also gets a `NOW:` line with the text that stands there. `NOW: -` means that the run reads as it did.


In [ ]:
#| export
def _changes(docs, answers):
    "The `answers` that rewrite a sentence of `docs` or ask about one"
    ss = [s for _,ps in docs for s,_ in ps if s]
    return [(a, b, t) for a,b,t in answers if 0 <= a <= b < len(ss) and t not in ('-', ' '.join(s.text for s in ss[a:b+1]))]

def _rng(a, b): return str(a) if a == b else f'{a}-{b}'

def _same(t): return re.sub(r'\W+', '', t).lower()

`_now` finds what stands in the edited file where each run of original sentences was. It lines up the two lists of sentences with `difflib` and returns None for a run that reads as it did.

In [ ]:
#| export
def _now(a, b, spans):
    "For each (first, last) of `spans` in sentence list `a`, the text that stands there in list `b`, or None when it is unchanged"
    ops,res = SequenceMatcher(None, a, b, autojunk=False).get_opcodes(),[]
    for x,y in spans:
        new,chg = [],False
        for tag,i,j,k,l in ops:
            if j <= x or i > y: continue
            if tag == 'equal': new += b[k+max(i, x)-i:k+min(j, y+1)-i]
            else: new,chg = new+b[k:l],True
        res.append(' '.join(new) if chg else None)
    return res

`compare` builds the text:

In [ ]:
#| export
def _cell_code(docs, k):
    "The index in `docs` of the code cell that doc `k` explains: the one a docstring is in, or the next one below prose"
    rng = range(k, len(docs)) if docs[k][0] == '' else range(k, -1, -1)
    return next((i for i in rng if docs[i][0] == CODE), None)

def compare(
    docs, # (name, parts) pairs of the original, as `file_docs` returns
    answers, # Each model's name with its answers, as `parse_rewrites` returns them
    now=None, # The sentences of the file after editing
    code=False, # Show a notebook's code cell under the first run that explains it?
    brief=False, # Keep only the runs that a model asked about or that every model changed?
    pending=False, # Keep only the runs that read as they did before editing? Needs `now`
):
    "Text for review: counts, then each run of sentences in `docs` that a model changed, with its address, the original, each answer, and with `now` what stands there after editing"
    ks,ss = [k for k,(_,ps) in enumerate(docs) for s,_ in ps if s],[s for _,ps in docs for s,_ in ps if s]
    ch = {nm.upper(): _changes(docs, a) for nm,a in answers.items()}
    spans,shown = [],set()
    for a,b in sorted({o[:2] for c in ch.values() for o in c}):
        if spans and a <= spans[-1][1]: spans[-1][1] = max(b, spans[-1][1])
        else: spans.append([a, b])
    cur = _now([s.text for s in ss], now, spans) if now is not None else None
    hdr = [f'{len(ss)} sentences.'] + [f"{nm}: {sum(t[0] != '?' for *_,t in c)} rewritten, {sum(t[0] == '?' for *_,t in c)} asked." for nm,c in ch.items()]
    if cur: hdr.append(f'Edited: {sum(o is not None for o in cur)} of {len(spans)} runs.')
    res = [' '.join(hdr)]
    for n,(a,b) in enumerate(spans):
        addr = getattr(ss[a], 'addr', None)
        rows,grp = [f"{_rng(a, b)} {f'@{addr} ' if addr else ''}ORIGINAL: {' '.join(s.text for s in ss[a:b+1])}"],{}
        for nm,c in ch.items():
            for x,y,t in c:
                if a <= x <= b: grp.setdefault((x, y, _same(t)), (t, []))[1].append(nm)
        rows += [f"  {', '.join(nms)} [{_rng(x, y)}]: {t}" for (x,y,_),(t,nms) in grp.items()]
        if kept := [nm for nm,c in ch.items() if not any(a <= x <= b for x,*_ in c)]: rows.append(f"  {', '.join(kept)}: -")
        if brief and kept and not any(t[0] == '?' for t,_ in grp.values()): continue
        if pending and cur[n] is not None: continue
        if cur: rows.append(f"  NOW: {cur[n] or '-'}")
        if code and (i := _cell_code(docs, ks[a])) is not None and i not in shown:
            shown.add(i)
            rows.append(textwrap.indent(docs[i][1][0][1], '  '))
        res.append('\n'.join(rows))
    if brief or pending: res[0] += f' Shown: {len(res)-1} of {len(spans)} runs.'
    return '\n\n'.join(res)

Here a second model answers the same file as `ans` above. It joins sentences 1 and 2, keeps sentence 7 and repeats sentence 4 unchanged, which counts as keeping it:

In [ ]:
ans2 = parse_rewrites("""1-2 `make` builds a widget and `ship` posts it to the customer.
4 Call `ship`.
7 -""")
cmp = compare(docs, dict(first=ans, second=ans2))
assert '1-2 ORIGINAL: `make` builds a widget. Finished widgets ride to the customer with `ship`.\n  FIRST [2]: Call `ship` to post' in cmp
assert '  SECOND [1-2]: `make` builds a widget and `ship` posts it to the customer.' in cmp
assert '7 ORIGINAL: The label carries the order number.\n  FIRST [7]: The order number is printed on the label.\n  SECOND: -' in cmp
assert '4 ORIGINAL' not in cmp
print(cmp)

A long comparison is quicker to read in two passes. `brief=True` keeps only the runs that a model asked a question about or that every model changed. Here the second model kept sentence 7, so that run is left out:

In [ ]:
short = compare(docs, dict(first=ans, second=ans2), brief=True)
assert short.startswith('9 sentences.') and 'Shown: 2 of 3 runs.' in short and '7 ORIGINAL' not in short
print(short)

A file's review files go to one folder, which is `review_dir(path)` under XDG state unless you pass `out`. The folder keeps the `orig` copy, which is the file before its first review. Each review then writes to a numbered folder of its own inside that folder, `1` for the first review. A later review therefore keeps the replies of the earlier ones. A numbered folder holds the `reviewed` copy, which is the file as that review numbered it, and the input, the replies and the comparison.

In [ ]:
#| export
def review_dir(path):
    "The folder under XDG state that holds the review files for the file at `path`"
    return xdg_state_home()/'slopometer'/'rewrites'/str(Path(path).expanduser().resolve().with_suffix('')).lstrip('/')

def _seen(d, path): return d/f'{path.stem}.reviewed{path.suffix}'

def _passes(path, out):
    "The numbered folders in `out` that each hold one review of the file at `path`, oldest first. Review files that an older version left in `out` itself are first moved into a new numbered folder."
    if not out.exists(): return []
    orig = out/f'{path.stem}.orig{path.suffix}'
    ds = sorted((d for d in out.iterdir() if d.name.isdigit() and _seen(d, path).exists()), key=lambda d: int(d.name))
    if old := [f for f in out.glob(f'{path.stem}.*') if f.is_file() and f != orig]:
        ds.append(d := out/str(int(ds[-1].name)+1 if ds else 1))
        d.mkdir(exist_ok=True)
        for f in old: f.rename(d/f.name)
        if not _seen(d, path).exists(): _seen(d, path).write_bytes(orig.read_bytes())
    return ds

def _review(path, out=None, n=None):
    "`path` as a `Path`, its review folder, the original that its first review saved there, and the folder of its review `n`: the latest if `n` is None, and None when it has no review"
    path = Path(path).expanduser()
    out = Path(out).expanduser() if out else review_dir(path)
    ds = _passes(path, out)
    if n is not None and not -len(ds) <= n-(n > 0) < len(ds): raise ValueError(f'{path} has {len(ds)} reviews in {out}, so it has no review {n}')
    return path, out, out/f'{path.stem}.orig{path.suffix}', ds[-1 if n is None else n-(n > 0)] if ds else None

`_answers` reads the replies that one review saved. `_numbers_ok` checks that this version of the package numbers the file's sentences as that review did, because a reply's numbers mean nothing otherwise:

In [ ]:
#| export
def _answers(d, path):
    "Each model's name with its answers, from the replies that the review in folder `d` saved for the file at `path`"
    pat = re.compile(re.escape(path.stem) + r'\.([^.]+)\.answer\.txt')
    return {m[1]: parse_rewrites(f.read_text()) for f in sorted(d.iterdir()) if (m := pat.fullmatch(f.name))}

def _marks(d, path):
    "The mark before each sentence in the input that the models read in the review in folder `d`: its number, or `-` for a sentence they were not asked about; None when that review saved no input"
    f = d/f'{path.stem}.input.txt'
    return re.findall(r'(?m)^(?:> |[-*+] |\d+[.)] )?\[(\d+|-)\] ', f.read_text()) if f.exists() else None

def _numbers_ok(path, d, n):
    "Raise `ValueError` unless the input that the models read in the review in folder `d` showed `n` sentences, as this version of the package does"
    seen = n if (ms := _marks(d, path)) is None else len(ms)
    if seen != n: raise ValueError(f'Review {d.name} of {path} numbered {seen} sentences and this version finds {n}: run `rewrite_review` on it again, or key the edits by their old text')

`rewrite_review` asks each model in `asks` about one file, all at once, with no system message. `ASKS` names the two models we use and the reasoning effort for each. It changes nothing in the file itself. It saves a copy of the file, the numbered input, each model's reply as returned, and the comparison, which it also returns. Give it a list of files and it reviews them together and returns one comparison for each.

`review_diff` compares the file with the `orig` copy, and a later review of the same file leaves that copy alone unless you pass `fresh=True`. When the file differs from that copy, a later review also gives the copy to the models as the earlier version, so that they can answer `? lost:` for a fact that an edit dropped. Pass `earlier` to name another file, such as the original of a file that you edited before its first review. The sentence numbers in `review_apply` and `review_report` are those of the latest review, unless you pass `n` to name an earlier one.

`fills`, `skip` and `mod_only` go to `rewrite_input`. Pass `mod_only=True` for a pyskill. Pass `context` to give the models files to consult, such as the code that a Markdown document describes. Their contents follow the document in the prompt. A long comparison is quicker to read in two passes. `brief=True` returns only the runs that a model asked a question about or that every model changed, and the whole comparison is still written to the folder. `code=True` prints each notebook code cell under the first run that explains it.


In [ ]:
#| export
ASKS = dict(carbon=('gemini/models/gemini-4-carbon', 'low'), sol=('gpt-6.1-sol', 'medium'))

def _save_orig(path, orig, last, fresh=False):
    "Save the file at `path` as `orig` when there is none, or when `fresh`, which first moves the present one into `last`, the folder of the latest review"
    if orig.exists() and not fresh: return
    if orig.exists() and last: orig.replace(last/orig.name)
    orig.write_bytes(path.read_bytes())

async def rewrite_review(
    path, # File to review, or a list of them: Markdown, a notebook, or a `.py` file for its docstrings
    out=None, # Folder to write to; `review_dir(path)` if None
    template=None, # Prompt with a `{document}` placeholder; `prompt()` if None
    asks=ASKS, # Name of each answer and its (model, effort)
    fills=None, # Text that each `{name}` placeholder in the sentences stands for
    context=None, # Files that the models can consult, such as the code that a document describes
    code=False, # Show each notebook code cell under the first run that explains it?
    brief=False, # Return only the runs that a model asked about or that every model changed?
    fresh=False, # Save the file as a new original? The original that it replaces moves into the folder of the latest review
    skip=None, # Regex; a paragraph or list item that it matches gets `[-]` in place of each number
    earlier=None, # File that holds an earlier version, for `? lost:` answers; the original that the first review saved if None and the file has changed since
    mod_only=False, # Give `[-]` to the docstrings of functions and classes? For a pyskill, whose module docstring is the skill's text
):
    "Ask each model in `asks` about the file at `path`; write the original to `out`, and a copy of the file, the input, each reply and their `compare` text to a new numbered folder in `out`; return that text, or one text for each file of a list; `brief` shortens what is returned, not what is written"
    if is_listy(path): return await asyncio.gather(*[rewrite_review(p, out, template, asks, fills, context, code, brief, fresh, skip, earlier, mod_only) for p in path])
    path,out,orig,last = _review(path, out)
    _,docs,_ = file_docs(path)
    if earlier is None and orig.exists() and not fresh and orig.read_bytes() != path.read_bytes(): earlier = orig
    replies = await asyncio.gather(*[ask_reply(docs, template, m, e, None, fills, context, skip, earlier, mod_only) for m,e in asks.values()])
    (d := out/str(int(last.name)+1 if last else 1)).mkdir(parents=True, exist_ok=True)
    for nm,r in zip(asks, replies): (d/f'{path.stem}.{nm}.answer.txt').write_text(r)
    _save_orig(path, orig, last, fresh)
    _seen(d, path).write_bytes(path.read_bytes())
    (d/f'{path.stem}.input.txt').write_text(rewrite_input(docs, fills, skip, mod_only=mod_only))
    ans = {nm: parse_rewrites(r) for nm,r in zip(asks, replies)}
    (d/f'{path.stem}.compare.txt').write_text(res := compare(docs, ans, code=code))
    return compare(docs, ans, code=code, brief=True) if brief else res

With `GEMINI_API_KEY` and `OPENAI_API_KEY` set, review one file:

In [ ]:
#| eval: false
print(await rewrite_review(root/'README.md'))


Here the two sets of answers above have reviewed a file, and one sentence has since been edited by hand. The review folder holds the original. Its folder `1` holds the first review's copy of the file and both replies, as `rewrite_review` would have written them:

In [ ]:
f,out = Path(tempfile.mkdtemp())/'widgets.py',Path(tempfile.mkdtemp())
(out/'widgets.orig.py').write_text(SRC)
(d1 := out/'1').mkdir()
(d1/'widgets.reviewed.py').write_text(SRC)
(d1/'widgets.first.answer.txt').write_text(reply)
(d1/'widgets.second.answer.txt').write_text('1-2 `make` builds a widget and `ship` posts it to the customer.\n7 the order number is printed on the label')
f.write_text(SRC.replace('The label carries the order number.', 'The order number is printed on the label.'))


`review_apply` makes the edits for you. Give it the new text for each sentence, keyed by the sentence's number in the comparison, by a (first, last) pair for a run, or by the old text itself. Numbers are those of the latest review, or of review `n`, and they still work after earlier edits have moved the sentences. An old text can be one sentence or a run of neighbouring ones, and it must occur exactly once. The call finds the cell and line itself. It writes nothing unless every edit can be made: a number that names a sentence you have already edited, an old text that is missing, and an edit that would change code each raise a `ValueError` that lists every failure. A notebook is validated before it is written. Numbers from a review that an older version of the package made can name the wrong sentences. The call refuses those too, and old texts still work as keys.

In [ ]:
#| export
def _sents(path): return [s.text for _,ps in file_docs(path)[1] for s,_ in ps if s]

def review_apply(
    path, # File to edit
    edits, # New text for each sentence or run, or an empty one to delete it; a key is a number or (first, last) pair from the comparison, or the old text itself
    out=None, # Folder that `rewrite_review` wrote to; `review_dir(path)` if None
    n=None, # Review whose numbers the keys use, counting from 1; the latest if None
):
    "Put each new text of `edits` into the file at `path` in place of the sentences that its key names; if any edit cannot be made, raise `ValueError` that lists them all, and write nothing"
    path,out,_,d = _review(path, out, n)
    src,docs,valid = file_docs(path)
    cur = [s.text for _,ps in docs for s,_ in ps if s]
    old = _sents(_seen(d, path)) if d else cur
    if d and not all(isinstance(k, str) for k in edits): _numbers_ok(path, d, len(old))
    at = {x: y for tag,i,j,k,l in SequenceMatcher(None, old, cur, autojunk=False).get_opcodes() if tag == 'equal' for x,y in zip(range(i, j), range(k, l))}
    ans,bad = [],[]
    for key,t in edits.items():
        if isinstance(key, str):
            want = ' '.join(key.split())
            hits = [(i, j) for i in range(len(cur)) for j in range(i, min(i+8, len(cur))) if ' '.join(cur[i:j+1]) == want]
            if len(hits) == 1: ans.append((*hits[0], t))
            else: bad.append(f'{key!r}: found {len(hits)} times')
        else:
            a,b = (key, key) if isinstance(key, int) else key
            if all(n in at for n in range(a, b+1)) and at[b]-at[a] == b-a: ans.append((at[a], at[b], t))
            else: bad.append(f'{_rng(a, b)}: not in the review, or edited since')
    rw = Rewrites(src, docs, ans, valid=valid)
    bad += [f'{o!r}: crosses a paragraph, overlaps another edit, or changes the code' for *_,kind,o,_ in rw.rows if kind == 'LEFT OUT']
    if bad: raise ValueError('No edit was written.\n' + '\n'.join(bad))
    write_new(path, rw.new)
    return f"{len(rw.of('AFTER'))} of {len(edits)} edits changed {path}"

Here one call replaces the run of sentences 1 and 2, using their numbers from the comparison:

In [ ]:
msg = review_apply(f, {(1, 2): '`make` builds a widget and `ship` posts it to the customer.'}, out)
assert '`make` builds a widget and `ship` posts it to the customer.\n' in f.read_text()
msg

Sentence 7 was edited by hand above, so its number no longer names a sentence in the file. The call fails, and the edit beside it is not written either:

In [ ]:
before = f.read_text()
test_fail(lambda: review_apply(f, {7: 'Labels have order numbers.', 'Call `ship`.': 'Then call `ship`.'}, out), contains='7: not in the review, or edited since')
test_eq(f.read_text(), before)

An empty new text deletes the sentences that its key names. `review_apply` also removes the space between them and the next sentence of their paragraph, or else the space after the previous sentence. It removes space on the same line first, which leaves each aligned row on its own line. When a paragraph or list item loses every sentence, `review_apply` removes its lines and leaves no extra blank line. This file has no review. Its keys are numbers in the file as it stands:

In [ ]:
MDX = 'Alpha one. Beta two. Gamma three.\nDelta four.\n\n- Item one.\n- Item two.\n  Item two more.\n\nx   row one. row two.\ny   row three.\n'
md = Path(tempfile.mkdtemp())/'notes.md'
def cut(*ks):
    md.write_text(MDX)
    review_apply(md, {k: '' for k in ks}, md.parent/'reviews')
    return md.read_text()
test_eq(cut(1).splitlines()[0], 'Alpha one. Gamma three.')
test_eq(cut(0).splitlines()[0], 'Beta two. Gamma three.')
test_eq(cut(2, 3), MDX.replace(' Gamma three.\nDelta four.', ''))
test_eq(cut(4), MDX.replace('- Item one.\n', ''))
test_eq(cut(5), MDX.replace('Item two.\n  ', ''))
test_eq(cut(8), MDX.replace(' row two.', ''))
test_eq(cut(7, 8), MDX.replace('x   row one. row two.\n', ''))
test_eq(cut(4, 5, 6), MDX.replace('- Item one.\n- Item two.\n  Item two more.\n\n', ''))
P = 'Para one.\n\nB.\n\nPara three.\n'
test_eq([Rewrites(P, [('', md_parts(P))], [(k, k, '')]).new for k in range(3)], ['B.\n\nPara three.\n', 'Para one.\n\nPara three.\n', 'Para one.\n\nB.\n'])
print(cut(2, 3, 5))

`review_diff` lists what you changed, by hand or with `review_apply`. It compares the file with the original that the first review saved.

In [ ]:
#| export
def _prose(path):
    "Each sentence, heading and other part of the prose in the file at `path` as (text, address, note); the note says when the text is a docstring or in an exported Markdown cell"
    path,res = Path(path),[]
    cells = {c.id: c for c in read_nb(path).cells} if path.suffix == '.ipynb' else {}
    for nm,ps in file_docs(path)[1]:
        if nm == CODE: continue
        for s,t in ps:
            addr = getattr(s, 'addr', None)
            c = cells.get(addr.split(':')[0]) if addr else None
            note = 'module docstring' if nm is None else f'docstring of `{nm}`' if nm else 'exported Markdown cell' if c and 'export' in c.directives else ''
            res.append((s.text if s else t, addr, note))
    return res

def _hunks(a, b):
    "Old texts `a` and new texts `b` of one changed stretch as (olds, news) pairs: a new pair starts at each old text that still has a similar new text"
    def sim(o, j): return SequenceMatcher(None, o, b[j]).ratio()
    cuts,k = [],0
    for i,o in enumerate(a):
        j = max(range(k, len(b)), key=partial(sim, o), default=None)
        if j is not None and sim(o, j) > 0.4: cuts,k = cuts+[(i, j)],j+1
    cuts = [(0, 0)] + cuts[1:] + [(len(a), len(b))]
    return [(a[i:x], b[j:y]) for (i,j),(x,y) in itertools.pairwise(cuts)]

def review_diff(
    path, # File that was reviewed and then edited
    out=None, # Folder that `rewrite_review` wrote to; `review_dir(path)` if None
    n=None, # Show only what changed during review `n`, counting from 1; every change since the first review if None
):
    "What differs between the original that the first review saved and the file at `path` now, or with `n` what changed during one review: the file's name, then each changed sentence or heading under its address, as `-` lines and the `+` lines that replace them"
    path,out,orig,d = _review(path, out, n)
    a,b = orig,path
    if n is not None:
        ds = _passes(path, out)
        a,b = _seen(d, path),next((_seen(o, path) for o in ds[ds.index(d)+1:]), path)
    a,b = _prose(a),_prose(b)
    res = []
    for tag,i,j,k,l in SequenceMatcher(None, [o[0] for o in a], [o[0] for o in b], autojunk=False).get_opcodes():
        if tag == 'equal': continue
        for olds,news in _hunks([o[0] for o in a[i:j]], [o[0] for o in b[k:l]]):
            _,addr,note = b[k] if news else a[i]
            hd = ' '.join(([f'@{addr}'] if addr else []) + ([f'({note})'] if note else []))
            res.append('\n'.join(([hd] if hd else []) + [f'- {o}' for o in olds] + [f'+ {o}' for o in news or ['(removed)']*bool(olds)]))
            i,k = i+len(olds),k+len(news)
    return '\n\n'.join([f"# {path}: {len(res)} change{'s'*(len(res) != 1)}{f' during review {d.name}' if n is not None else ''}"] + res)

The diff starts with the file's name and the number of changes. Each change has the address of the text as it now stands, and a note when the text is a docstring or in an exported Markdown cell, since a module generated from the file then needs exporting again. The two sentences that became one are shown together:

In [ ]:
dif = review_diff(f, out)
assert dif.startswith(f'# {f}: 2 changes')
assert '(module docstring)\n- `make` builds a widget.\n- Finished widgets ride to the customer with `ship`.\n+ `make` builds a widget and `ship` posts it to the customer.' in dif
assert f'@{lnhash_at(f.read_text(), 21)} (docstring of `Widget.ship`)\n- The label carries the order number.\n+ The order number is printed on the label.' in dif
print(dif)

`review_report` writes the comparison again with a `NOW:` line for each run, which shows the answers you took and those you left. It reports on the latest review, or on review `n`. It reads that review's copy of the file and its replies, and writes the report beside them. It returns the counts and the report's file name, because the whole text is long. Pass `full=True` to get the text.

In [ ]:
#| export
def review_report(
    path, # File that was reviewed and then edited
    out=None, # Folder that `rewrite_review` wrote to; `review_dir(path)` if None
    full=False, # Return the whole report, not only its counts and where it is?
    code=False, # Show each notebook code cell under the first run that explains it?
    brief=False, # Keep only the runs that a model asked about or that every model changed?
    n=None, # Review to report on, counting from 1; the latest if None
):
    "Write the `compare` text of one review of the file at `path`, with what each run of sentences reads after editing, to that review's folder; return its counts and the file's name, or the text with `full`"
    path,out,_,d = _review(path, out, n)
    if not d: raise ValueError(f'{path} has no review in {out}')
    _numbers_ok(path, d, len(_sents(_seen(d, path))))
    res = compare(file_docs(_seen(d, path))[1], _answers(d, path), _sents(path), code, brief)
    (f := d/f'{path.stem}.report.txt').write_text(res)
    return res if full else f'{res.splitlines()[0]}\nFull report: {f}'

The report starts with the counts. Sentence 7 is on line 22 of the file, both models gave the same rewrite for it, and it was taken. So was the second model's answer for sentences 1 and 2. The last run reads as it did:

In [ ]:
rep = review_report(f, out, full=True)
assert rep.startswith('9 sentences. FIRST: 2 rewritten, 1 asked. SECOND: 2 rewritten, 0 asked. Edited: 2 of 3 runs.')
assert f"7 @{lnhash_at(SRC, 21)} ORIGINAL: The label carries the order number.\n  FIRST, SECOND [7]: The order number is printed on the label.\n  NOW: The order number is printed on the label." in rep
test_eq(rep.count('  NOW: -'), 1)
print(rep)
review_report(f, out)


Each review has a numbered folder of its own, so a second review keeps the replies of the first. Here a second review of the edited file has one reply with two rewrites, and `review_apply` puts in one of them:

In [ ]:
(d2 := out/'2').mkdir()
(d2/'widgets.reviewed.py').write_text(f.read_text())
(d2/'widgets.third.answer.txt').write_text('0 Widgets for a shop floor.\n7 `ship` raises `ValueError` when `addr` is empty.')
review_apply(f, {7: '`ship` raises `ValueError` when `addr` is empty.'}, out)

`review_diff` with `n` shows what changed during one review. `n` counts the reviews from 1, and a negative `n` counts back from the latest:

In [ ]:
dif2 = review_diff(f, out, n=2)
assert dif2.startswith(f'# {f}: 1 change during review 2') and 'order number' not in dif2
assert '2 changes during review 1' in review_diff(f, out, n=1)
test_eq(review_diff(f, out, n=-1), dif2)
print(dif2)

`review_pending` goes through every review. For each one it lists the runs that a model rewrote or asked about and that still read as they did in that review. Use it to find suggestions that nobody has acted on.

In [ ]:
#| export
def review_pending(
    path, # File that has been reviewed
    out=None, # Folder that `rewrite_review` wrote to; `review_dir(path)` if None
    code=False, # Show each notebook code cell under the first run that explains it?
):
    "For each review of the file at `path`, the runs that a model rewrote or asked about and that still read as they did in that review: one `compare` text under a heading for each review"
    path,out,_,_ = _review(path, out)
    now,res = _sents(path),[]
    for d in _passes(path, out):
        try: _numbers_ok(path, d, len(_sents(_seen(d, path))))
        except ValueError as e: txt = str(e)
        else: txt = compare(file_docs(_seen(d, path))[1], _answers(d, path), now, code, pending=True)
        res.append(f'# Review {d.name}\n{txt}')
    return '\n\n'.join(res) or f'{path} has no review in {out}'

Every run of the first review has been edited since. In the second review, nobody has acted on the rewrite of sentence 0:

In [ ]:
pend = review_pending(f, out)
r1,r2 = pend.split('# Review 2\n')
assert r1.startswith('# Review 1\n') and 'Shown: 0 of 3 runs.' in r1
assert 'Shown: 1 of 2 runs.' in r2 and 'ORIGINAL: Widgets for the shop floor.' in r2 and 'addr' not in r2
print(pend)

The models miss sentences. A docstring can read clearly beside the prose that explains it and still be unclear on its own. `review_passed` lists the sentences of one review that no model rewrote or asked about, without the code, for a person or an agent to read. It leaves out the sentences that the models read without a number, which `mod_only` and `skip` mark `[-]`. Each has the number that `review_apply` takes and its address. A line names each docstring above its sentences.

In [ ]:
#| export
def review_passed(
    path, # File that has been reviewed
    out=None, # Folder that `rewrite_review` wrote to; `review_dir(path)` if None
    n=None, # Review to read, counting from 1; the latest if None
):
    "The numbered sentences that no model rewrote or asked about in one review of the file at `path`: a count, then each as its number, address and text, with a line that names each docstring"
    path,out,_,d = _review(path, out, n)
    if not d: raise ValueError(f'{path} has no review in {out}')
    docs = file_docs(_seen(d, path))[1]
    _numbers_ok(path, d, sum(1 for _,ps in docs for s,_ in ps if s))
    skip = {k for k,m in enumerate(_marks(d, path) or []) if m == '-'}
    hit = {i for c in _answers(d, path).values() for a,b,_ in _changes(docs, c) for i in range(a, b+1)} | skip
    res,i,tot = [],0,0
    for nm,ps in docs:
        rows = []
        for s in (s for s,_ in ps if s):
            if i not in hit: rows.append(f"{i} {f'@{s.addr} ' if getattr(s, 'addr', None) else ''}{s.text}")
            i += 1
        if rows: res.append('\n'.join(([] if nm == '' else ['Module docstring:' if nm is None else f'Docstring of `{nm}`:']) + rows))
        tot += len(rows)
    return '\n\n'.join([f'{tot} of {i-len(skip)} sentences that no model changed in review {d.name}.'] + res)

In the first review, the two models between them rewrote or asked about sentences 1, 2, 7 and 8. The other five are listed:

In [ ]:
psd = review_passed(f, out, n=1)
assert psd.startswith('5 of 9 sentences that no model changed in review 1.')
assert 'Docstring of `make`:\n5 @' in psd and 'Build a widget from `spec`' in psd
assert 'order number' not in psd and 'def make' not in psd
print(psd)

In [ ]:
#| hide
(inp := d1/'widgets.input.txt').write_text(rewrite_input(py_docs(SRC), mod_only=True))
mpsd = review_passed(f, out, n=1)
assert mpsd.startswith('3 of 5 sentences') and 'Build a widget' not in mpsd
inp.unlink()

`fresh=True` makes `rewrite_review` save the file as a new original. `_save_orig` first moves the original that it replaces into the folder of the latest review, so that no saved text is lost. Here the file's second review is the latest:

In [ ]:
orig = out/'widgets.orig.py'
_save_orig(f, orig, out/'2', fresh=True)
test_eq((out/'2'/'widgets.orig.py').read_text(), SRC)
test_eq(orig.read_text(), f.read_text())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()